# 01 - Data Exploration

This notebook explores the scraped `books.csv` dataset and provides descriptive analytics and visualizations.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

In [ ]:
base_dir = Path('.').resolve()
csv_path = base_dir / 'books.csv'
if not csv_path.exists():
    raise FileNotFoundError(f'Missing books.csv at {csv_path}')

df = pd.read_csv(csv_path)

for col in ['title', 'authors', 'genre', 'publisher', 'subjects', 'description', 'image_url']:
    if col not in df.columns:
        df[col] = ''

for col in ['publish_year', 'ratings_avg', 'pages', 'ratings_count', 'want_to_read_count', 'edition_count']:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df.head()

## Section 1 - Dataset Overview

In [ ]:
total_books = len(df)
unique_authors = df['authors'].fillna('').astype(str).str.split(';').explode().str.strip()
unique_authors = unique_authors[unique_authors != ''].nunique()
unique_genres = df['genre'].fillna('').astype(str).nunique()
unique_publishers = df['publisher'].fillna('').astype(str).nunique()

year_min = int(df['publish_year'].replace(0, np.nan).min()) if (df['publish_year'] > 0).any() else None
year_max = int(df['publish_year'].max()) if len(df) else None
desc_pct = (df['description'].fillna('').astype(str).str.len() > 0).mean() * 100
img_pct = (df['image_url'].fillna('').astype(str).str.len() > 0).mean() * 100

print(f'Total books: {total_books:,}')
print(f'Unique authors: {unique_authors:,}')
print(f'Unique genres: {unique_genres:,}')
print(f'Unique publishers: {unique_publishers:,}')
print(f'Year range: {year_min} - {year_max}')
print(f'% books with description: {desc_pct:.2f}%')
print(f'% books with image URL: {img_pct:.2f}%')

In [ ]:
preview_cols = ['book_id', 'title', 'authors', 'genre', 'publish_year', 'image_url']
preview_cols = [c for c in preview_cols if c in df.columns]
preview = df[preview_cols].head(5).copy()
if 'image_url' in preview.columns:
    preview['image_url'] = preview['image_url'].apply(lambda x: f'<a href="{x}" target="_blank">{x}</a>' if isinstance(x, str) and x else '')
preview.style.format(escape='html')

## Section 2 - Distribution Analysis

In [ ]:
genre_counts = df['genre'].fillna('Unknown').value_counts()
plt.figure(figsize=(14, 6))
sns.barplot(x=genre_counts.index, y=genre_counts.values)
plt.xticks(rotation=60, ha='right')
plt.title('Book Count per Genre')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['publish_year'][df['publish_year'] >= 2020], bins=20, kde=False, ax=axes[0, 0])
axes[0, 0].set_title('Publish Year Distribution (2020+)')

sns.histplot(df['ratings_avg'], bins=30, kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Ratings Average Distribution')

page_cap = df['pages'].quantile(0.99) if len(df) else 0
sns.histplot(df['pages'].clip(upper=page_cap), bins=30, kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Pages Distribution (Capped at 99th Percentile)')

ratings_count_log = np.log1p(df['ratings_count'])
sns.histplot(ratings_count_log, bins=30, kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Ratings Count Distribution (Log Scale)')
axes[1, 1].set_xlabel('log(1 + ratings_count)')

plt.tight_layout()
plt.show()

## Section 3 - Genre Deep Dive

In [ ]:
pivot = pd.pivot_table(
    df,
    index='genre',
    columns='publish_year',
    values='book_id' if 'book_id' in df.columns else 'title',
    aggfunc='count',
    fill_value=0
)

plt.figure(figsize=(14, 8))
sns.heatmap(pivot, cmap='YlGnBu')
plt.title('Genre vs Publish Year (Count Heatmap)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='genre', y='ratings_avg')
plt.xticks(rotation=60, ha='right')
plt.title('Ratings Average per Genre')
plt.tight_layout()
plt.show()

In [ ]:
all_subjects = (
    df['subjects']
    .fillna('')
    .astype(str)
    .str.split(';')
    .explode()
    .str.strip()
)
subject_counts = all_subjects[all_subjects != ''].value_counts().head(20)

plt.figure(figsize=(14, 6))
sns.barplot(x=subject_counts.values, y=subject_counts.index)
plt.title('Top 20 Most Common Subjects/Tags')
plt.xlabel('Count')
plt.ylabel('Subject')
plt.tight_layout()
plt.show()

## Section 4 - Author & Publisher Analysis

In [ ]:
authors_series = (
    df['authors']
    .fillna('')
    .astype(str)
    .str.split(';')
    .explode()
    .str.strip()
)
top_authors = authors_series[authors_series != ''].value_counts().head(15)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_authors.values, y=top_authors.index)
plt.title('Top 15 Most Prolific Authors')
plt.xlabel('Book Count')
plt.ylabel('Author')
plt.tight_layout()
plt.show()

In [ ]:
top_publishers = df['publisher'].fillna('Unknown').astype(str).value_counts().head(15)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_publishers.values, y=top_publishers.index)
plt.title('Top 15 Publishers by Book Count')
plt.xlabel('Book Count')
plt.ylabel('Publisher')
plt.tight_layout()
plt.show()

## Section 5 - Correlation & Feature Analysis

In [ ]:
sample_df = df.copy()
if len(sample_df) > 3000:
    sample_df = sample_df.sample(3000, random_state=42)

plt.figure(figsize=(12, 6))
sns.scatterplot(data=sample_df, x='ratings_count', y='ratings_avg', hue='genre', alpha=0.7, s=40)
plt.xscale('log')
plt.title('Ratings Count vs Ratings Average (Colored by Genre)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=sample_df, x='pages', y='ratings_avg', hue='genre', alpha=0.7, s=40)
plt.title('Pages vs Ratings Average')
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['pages', 'ratings_avg', 'ratings_count', 'want_to_read_count', 'edition_count', 'publish_year']
corr = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap of Numerical Features')
plt.tight_layout()
plt.show()

## Section 6 - Text Feature Analysis

In [ ]:
subjects_text = ' '.join(df['subjects'].fillna('').astype(str).tolist())
if subjects_text.strip():
    wc_subjects = WordCloud(width=1400, height=700, background_color='white', colormap='viridis').generate(subjects_text)
    plt.figure(figsize=(14, 7))
    plt.imshow(wc_subjects, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud - Subjects')
    plt.show()
else:
    print('No subjects text available for word cloud.')

In [ ]:
desc_text = ' '.join(df['description'].fillna('').astype(str).tolist())
if desc_text.strip():
    wc_desc = WordCloud(width=1400, height=700, background_color='white', colormap='magma').generate(desc_text)
    plt.figure(figsize=(14, 7))
    plt.imshow(wc_desc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud - Descriptions')
    plt.show()
else:
    print('No description text available for word cloud.')

In [ ]:
desc_len = df['description'].fillna('').astype(str).str.len()
length_bucket = pd.Series(np.where(desc_len == 0, 'missing', np.where(desc_len < 200, 'short', np.where(desc_len < 800, 'medium', 'long'))))
bucket_counts = length_bucket.value_counts().reindex(['missing', 'short', 'medium', 'long']).fillna(0)

plt.figure(figsize=(10, 5))
sns.barplot(x=bucket_counts.index, y=bucket_counts.values)
plt.title('Description Length Distribution')
plt.xlabel('Bucket')
plt.ylabel('Count')
plt.tight_layout()
plt.show()